# MDMP visualization gallery

This notebook demonstrates every plotting function in `mdmp.plotting` on the bundled
`mdmr_test_data` dataset, then composes a publication-ready 2×2 gallery figure
for the software paper (`figures/fig5_visualization-gallery.pdf`).

**Plotting API covered:** `plot_dag`, `plot_arcs`, `plot_marginal`, `plot_stream`, `plot_idag`.

In [ ]:
from __future__ import annotations

import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mdmp import MDM, load_dataset
from mdmp.plotting import plot_arcs, plot_dag, plot_idag, plot_marginal, plot_stream

# Paths: notebook lives in mdmp/notebooks/
REPO_ROOT = Path("..").resolve().parent
FIGURES_DIR = REPO_ROOT / "figures"
NOTEBOOK_OUT = Path("output")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

# Paul Tol–inspired palette (no yellow/green); colour-blind-friendly
C = {
    "blue": "#0077BB",
    "purple": "#AA3377",
    "rose": "#CC6677",
    "wine": "#882255",
    "sky": "#33BBEE",
    "ci_fill": "#99C2E0",
    "gray": "#666666",
    "lgray": "#BBBBBB",
}

# Panels (c) & (d): Okabe–Ito — same as figures/gen_figures.py fig5()
C["orange"] = "#E69F00"
C["green"] = "#009E73"
C["observed"] = C["gray"]
GALLERY_PARAM_COLORS = [C["orange"], "#0072B2", C["green"]]  # orange / blue / green
STREAM_ALPHA = 0.78

CI_ALPHA = 0.72  # panel (b) only

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.08,
})

In [ ]:
from matplotlib.collections import LineCollection, PathCollection, PolyCollection
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, Patch


def _param_legend_labels(model, node_idx: int) -> list[str]:
    """Human-readable labels from filtered posterior row names."""
    row_names = (model.Filt.get("row_names") or {}).get(node_idx, [])
    labels: list[str] = []
    for name in row_names:
        s = str(name)
        if "beta0" in s:
            labels.append("intercept")
        elif "->" in s:
            labels.append(s.replace("->", "→"))
        else:
            labels.append(s)
    return labels


from scipy import stats


def _arc_series(model, edge_label: str, *, distribution: str = "filt", ci_level: float = 0.95):
    """Posterior mean and CI half-width for one edge (same logic as ``plot_arcs``)."""
    parent, child = edge_label.split("->")
    child_idx = list(model.node_names).index(child)
    row_names = (model.Filt.get("row_names") or {}).get(child_idx, [])
    param_idx = next(i for i, n in enumerate(row_names) if str(n) == edge_label)

    if distribution == "filt":
        mt_node = model.Filt["mt"][child_idx]
        Ct_node = model.Filt["Ct"][child_idx]
        nt_node = model.Filt["nt"][child_idx]
    else:
        mt_node = model.Smoo["smt"][child_idx]
        Ct_node = model.Smoo["sCt"][child_idx]
        nt_node = model.Filt["nt"][child_idx]

    if mt_node.ndim == 1:
        mt_node = mt_node.reshape(1, -1)
    mean = mt_node[param_idx, :]
    if Ct_node.ndim == 3:
        var = Ct_node[param_idx, param_idx, :]
    else:
        var = Ct_node[param_idx, param_idx]
    half_width = stats.t.ppf((1 + ci_level) / 2, nt_node) * np.sqrt(var)
    time = np.arange(mean.shape[0])
    return time, mean, half_width


def draw_arc_panel(
    ax,
    model,
    edge_label: str,
    *,
    distribution: str = "filt",
    ci_level: float = 0.95,
    ci_alpha: float = CI_ALPHA,
) -> None:
    """Draw one ``plot_arcs`` connection with a visible credible-interval band."""
    time, mean, hw = _arc_series(model, edge_label, distribution=distribution, ci_level=ci_level)
    edge_tex = edge_label.replace("->", "→")
    ax.fill_between(
        time, mean - hw, mean + hw,
        color=C["ci_fill"], alpha=ci_alpha, zorder=1, linewidth=0,
        label=f"{ci_level*100:.0f}% CI",
    )
    ax.plot(time, mean, color=C["blue"], lw=2.2, label=edge_tex, zorder=3)
    ax.set_xlim(time[0], time[-1])
    ax.set_xlabel("Time  $t$")
    ax.set_ylabel("Coefficient")
    ax.axhline(0, color=C["gray"], lw=0.9, ls="--", alpha=0.7, zorder=0)
    ax.grid(True, alpha=0.25)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.legend(fontsize=9, loc="upper right", framealpha=0.9)


def _clone_lines_axis(src_ax, dst_ax) -> None:
    """Copy line plots and legend only (no polygon fills)."""
    for line in src_ax.get_lines():
        dst_ax.plot(
            line.get_xdata(), line.get_ydata(),
            color=line.get_color(), lw=line.get_linewidth(),
            label=line.get_label(), alpha=line.get_alpha(), zorder=line.get_zorder(),
        )
    dst_ax.set_xlim(src_ax.get_xlim())
    dst_ax.set_ylim(src_ax.get_ylim())
    if src_ax.get_xlabel():
        dst_ax.set_xlabel(src_ax.get_xlabel())
    if src_ax.get_ylabel():
        dst_ax.set_ylabel(src_ax.get_ylabel())
    dst_ax.grid(True, alpha=0.25)
    for spine in ("top", "right"):
        dst_ax.spines[spine].set_visible(False)
    handles, labels = src_ax.get_legend_handles_labels()
    if handles:
        dst_ax.legend(handles, labels, fontsize=9, loc="upper right", framealpha=0.9)


def _clone_stream_axis(src_ax, dst_ax) -> None:
    """Copy stacked-area polygons preserving face colours."""
    poly_count = 0
    for coll in src_ax.collections:
        if not isinstance(coll, PolyCollection):
            continue
        fc = coll.get_facecolor()
        alpha = coll.get_alpha() if coll.get_alpha() is not None else STREAM_ALPHA
        for i, path in enumerate(coll.get_paths()):
            color = fc[i % len(fc)] if len(fc) else GALLERY_PARAM_COLORS[0]
            v = path.vertices
            dst_ax.fill(v[:, 0], v[:, 1], color=color, alpha=alpha, edgecolor="none")
            poly_count += 1
    dst_ax.set_xlim(src_ax.get_xlim())
    dst_ax.set_ylim(src_ax.get_ylim())
    if src_ax.get_xlabel():
        dst_ax.set_xlabel(src_ax.get_xlabel())
    if src_ax.get_ylabel():
        dst_ax.set_ylabel(src_ax.get_ylabel())
    dst_ax.grid(True, alpha=0.2)
    for spine in ("top", "right"):
        dst_ax.spines[spine].set_visible(False)
    _, labels = src_ax.get_legend_handles_labels()
    if labels:
        dst_ax.legend(
            handles=[
                Patch(
                    facecolor=GALLERY_PARAM_COLORS[i % len(GALLERY_PARAM_COLORS)],
                    edgecolor="none",
                    alpha=STREAM_ALPHA,
                    label=labels[i] if i < len(labels) else f"param {i}",
                )
                for i in range(poly_count)
            ],
            fontsize=9,
            loc="upper right",
            framealpha=0.9,
        )


def _clone_dag_axis(src_ax, dst_ax) -> None:
    for coll in src_ax.collections:
        if isinstance(coll, PathCollection):
            offsets = coll.get_offsets()
            dst_ax.scatter(
                offsets[:, 0],
                offsets[:, 1],
                s=coll.get_sizes(),
                c=coll.get_facecolors(),
                edgecolors=coll.get_edgecolors(),
                linewidths=coll.get_linewidths(),
                zorder=2,
            )
    for patch in src_ax.patches:
        if isinstance(patch, FancyArrowPatch):
            pos_a, pos_b = patch._posA_posB
            dst_ax.add_patch(
                FancyArrowPatch(
                    pos_a,
                    pos_b,
                    arrowstyle=patch.get_arrowstyle(),
                    mutation_scale=patch.get_mutation_scale(),
                    linewidth=patch.get_linewidth(),
                    color=patch.get_edgecolor(),
                    zorder=1,
                )
            )
    for text in src_ax.texts:
        dst_ax.text(
            text.get_position()[0],
            text.get_position()[1],
            text.get_text(),
            ha=text.get_ha(),
            va=text.get_va(),
            fontsize=text.get_fontsize(),
            color=text.get_color(),
            fontweight=text.get_fontweight(),
            zorder=3,
        )
    dst_ax.set_xlim(src_ax.get_xlim())
    dst_ax.set_ylim(src_ax.get_ylim())
    dst_ax.axis("off")


def panel_label(ax, text: str, *, y: float = 1.02) -> None:
    ax.text(
        0.0,
        y,
        text,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="bottom",
        ha="left",
    )


def find_arc_axis(fig, edge_label: str):
    for ax in fig.axes:
        if ax.get_visible() and ax.get_title() == edge_label:
            return ax
    raise ValueError(f"Edge panel {edge_label!r} not found in plot_arcs output")


def single_arc_figure(model, edge_label: str, *, distribution: str = "filt", ci_level: float = 0.95):
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    draw_arc_panel(ax, model, edge_label, distribution=distribution, ci_level=ci_level)
    fig.tight_layout()
    return fig


def style_marginal_figure(fig, model, target_node: int):
    ax = fig.axes[0]
    param_labels = _param_legend_labels(model, target_node)
    lines = ax.get_lines()
    if lines:
        lines[0].set_color(C["observed"])
        lines[0].set_alpha(0.45)
        lines[0].set_linewidth(1.0)
        lines[0].set_label("observed")
        lines[0].set_zorder(1)
    for i, line in enumerate(lines[1:]):
        color = GALLERY_PARAM_COLORS[i % len(GALLERY_PARAM_COLORS)]
        line.set_color(color)
        line.set_linewidth(2.0)
        line.set_zorder(i + 2)
        if i < len(param_labels):
            line.set_label(param_labels[i])
    ax.axhline(0, color=C["gray"], lw=0.8, ls="--", alpha=0.6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel("Time  $t$")
    ax.set_ylabel("Parameter")
    ax.set_title("")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=9, loc="upper right", framealpha=0.9)
    return fig


def style_stream_figure(fig, model, child_node: int):
    ax = fig.axes[0]
    param_labels = _param_legend_labels(model, child_node)
    poly_cols = [c for c in ax.collections if isinstance(c, PolyCollection)]
    for i, coll in enumerate(poly_cols):
        color = GALLERY_PARAM_COLORS[i % len(GALLERY_PARAM_COLORS)]
        coll.set_facecolor(color)
        coll.set_alpha(STREAM_ALPHA)
        coll.set_edgecolor("none")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel("Time  $t$")
    ax.set_ylabel("Contribution")
    ax.set_title("")
    ax.grid(True, alpha=0.2)
    ax.legend(
        handles=[
            Patch(
                facecolor=GALLERY_PARAM_COLORS[i % len(GALLERY_PARAM_COLORS)],
                edgecolor="none",
                alpha=STREAM_ALPHA,
                label=param_labels[i] if i < len(param_labels) else f"param {i}",
            )
            for i in range(len(poly_cols))
        ],
        fontsize=9,
        loc="upper right",
        framealpha=0.9,
    )
    return fig


def best_gallery_node(model) -> int:
    n_parents = model.adj_mat.sum(axis=0)
    if n_parents.max() == 0:
        raise ValueError("No node with parents found in learned structure")
    return int(np.argmax(n_parents))

## Fit MDM on bundled test data

In [ ]:
data = load_dataset("mdmr_test_data")
print(f"Data shape: {data.shape}")
print(f"Variables: {list(data.columns)}")

model = MDM(data, method="hc", nbf=15, verbose=False)
print("\nLearned adjacency matrix (row = parent, col = child):")
print(model.adj_mat)

TARGET_NODE = best_gallery_node(model)
TARGET_NAME = model.node_names[TARGET_NODE]
parent_idx = np.where(model.adj_mat[:, TARGET_NODE] > 0)[0]
parent_names = [model.node_names[i] for i in parent_idx]
ARC_EDGE = f"{parent_names[0]}->{TARGET_NAME}"

print(f"\nGallery target node: {TARGET_NAME} (parents: {parent_names})")
print(f"Gallery arc panel: {ARC_EDGE}")

## All plotting functions

Each panel below is produced directly by the public `mdmp.plotting` API.

In [ ]:
# 1. plot_dag — graph view
fig_dag_graph = plot_dag(
    model,
    plot_type="graph",
    node_color=C["sky"],
    edge_color=C["blue"],
    layout_seed=5,
    figsize=(7, 6),
)
fig_dag_graph.axes[0].set_title("")
fig_dag_graph.savefig(NOTEBOOK_OUT / "01_plot_dag_graph.png", dpi=150, bbox_inches="tight")
fig_dag_graph

In [ ]:
# 2. plot_dag — heatmap view
fig_dag_heatmap = plot_dag(model, plot_type="heatmap", figsize=(6, 5.5))
fig_dag_heatmap.savefig(NOTEBOOK_OUT / "02_plot_dag_heatmap.png", dpi=150, bbox_inches="tight")
fig_dag_heatmap

In [ ]:
# 3. plot_arcs — dynamic connections (filtered)
fig_arcs_conn_filt = plot_arcs(model, plot_type="connections", distribution="filt", ci_level=0.95)
fig_arcs_conn_filt.savefig(NOTEBOOK_OUT / "03_plot_arcs_connections_filt.png", dpi=150, bbox_inches="tight")
fig_arcs_conn_filt

In [ ]:
# 4. plot_arcs — intercepts (filtered)
fig_arcs_int_filt = plot_arcs(model, plot_type="intercepts", distribution="filt")
fig_arcs_int_filt.savefig(NOTEBOOK_OUT / "04_plot_arcs_intercepts_filt.png", dpi=150, bbox_inches="tight")
fig_arcs_int_filt

In [ ]:
# 5. plot_arcs — connections (smoothed)
fig_arcs_conn_smoo = plot_arcs(model, plot_type="connections", distribution="smoo", ci_level=0.95)
fig_arcs_conn_smoo.savefig(NOTEBOOK_OUT / "05_plot_arcs_connections_smoo.png", dpi=150, bbox_inches="tight")
fig_arcs_conn_smoo

In [ ]:
# 6. plot_marginal — filtered posterior for target node
fig_marg_filt = style_marginal_figure(
    plot_marginal(model, target_node=TARGET_NODE, distribution="filt", figsize=(9, 4.5)),
    model,
    TARGET_NODE,
)
fig_marg_filt.savefig(NOTEBOOK_OUT / "06_plot_marginal_filt.png", dpi=150, bbox_inches="tight")
fig_marg_filt

In [ ]:
# 7. plot_marginal — smoothed posterior for target node
fig_marg_smoo = style_marginal_figure(
    plot_marginal(model, target_node=TARGET_NODE, distribution="smoo", figsize=(9, 4.5)),
    model,
    TARGET_NODE,
)
fig_marg_smoo.savefig(NOTEBOOK_OUT / "07_plot_marginal_smoo.png", dpi=150, bbox_inches="tight")
fig_marg_smoo

In [ ]:
# 8. plot_stream — filtered parent contributions
fig_stream_filt = style_stream_figure(
    plot_stream(model, child_node=TARGET_NODE, distribution="filt", figsize=(9, 4.5)),
    model,
    TARGET_NODE,
)
fig_stream_filt.savefig(NOTEBOOK_OUT / "08_plot_stream_filt.png", dpi=150, bbox_inches="tight")
fig_stream_filt

In [ ]:
# 9. plot_stream — smoothed parent contributions
fig_stream_smoo = style_stream_figure(
    plot_stream(model, child_node=TARGET_NODE, distribution="smoo", figsize=(9, 4.5)),
    model,
    TARGET_NODE,
)
fig_stream_smoo.savefig(NOTEBOOK_OUT / "09_plot_stream_smoo.png", dpi=150, bbox_inches="tight")
fig_stream_smoo

In [ ]:
# 10. plot_idag — animated dynamic-parameter heatmap (GIF)
from mdmp.plotting.animation import (
    _param_matrix_at_time,
    _resolve_node_labels,
    _row_names_for_idag,
)

gif_path = NOTEBOOK_OUT / "10_plot_idag.gif"
plot_idag(model, output_gif=str(gif_path), fps=8, distribution="filt")
print(f"Saved animation: {gif_path.resolve()}")

# Static mid-time snapshot for the catalog
mid_t = len(model.data) // 2
labels = _resolve_node_labels(model, model.adj_mat.shape[0])
row_names = _row_names_for_idag(model, "filt")
frame = _param_matrix_at_time(
    model, model.Filt["mt"], row_names, labels, mid_t, show_intercepts=False
)

fig_idag_snap, ax_idag = plt.subplots(figsize=(6, 5.5))
cmap = plt.cm.RdBu_r.copy()
cmap.set_bad(color="#f5f5f5")
finite = frame[np.isfinite(frame)]
vmin, vmax = float(finite.min()), float(finite.max())
im = ax_idag.imshow(frame, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax, origin="upper")
ax_idag.set_xticks(range(len(labels)))
ax_idag.set_yticks(range(len(labels)))
ax_idag.set_xticklabels(labels, rotation=45, ha="right")
ax_idag.set_yticklabels(labels)
ax_idag.set_title(f"plot_idag snapshot (t = {mid_t})")
fig_idag_snap.colorbar(im, ax=ax_idag, fraction=0.046, pad=0.04)
fig_idag_snap.savefig(NOTEBOOK_OUT / "10_plot_idag_snapshot.png", dpi=150, bbox_inches="tight")
fig_idag_snap

## Publication gallery (Figure 5)

Four-panel layout matching the software paper. Panels are drawn on native Matplotlib axes
(no rasterised embedding), so text and lines stay sharp in the PDF export.

Outputs: `figures/fig5_visualization-gallery.pdf` and `.png`.

In [ ]:
def build_publication_gallery(model, *, target_node: int, arc_edge: str):
    """Compose a 2×2 gallery with native axes (vector-friendly, no squashing)."""
    target_name = model.node_names[target_node]

    temp_dag = plot_dag(
        model,
        plot_type="graph",
        node_color=C["sky"],
        edge_color=C["blue"],
        layout_seed=5,
        figsize=(6.5, 5.5),
        node_size=1800,
        font_size=11,
        edge_width=2.0,
        arrow_size=3.5,
    )
    temp_marg = style_marginal_figure(
        plot_marginal(model, target_node=target_node, distribution="filt", figsize=(6.5, 4.0)),
        model,
        target_node,
    )
    temp_stream = style_stream_figure(
        plot_stream(model, child_node=target_node, distribution="filt", figsize=(6.5, 4.0)),
        model,
        target_node,
    )

    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(
        2, 2,
        hspace=0.42,
        wspace=0.34,
        left=0.07,
        right=0.97,
        top=0.94,
        bottom=0.08,
    )

    ax_dag = fig.add_subplot(gs[0, 0])
    ax_arc = fig.add_subplot(gs[0, 1])
    ax_marg = fig.add_subplot(gs[1, 0])
    ax_stream = fig.add_subplot(gs[1, 1])

    _clone_dag_axis(temp_dag.axes[0], ax_dag)
    draw_arc_panel(ax_arc, model, arc_edge, distribution="filt", ci_level=0.95)
    _clone_lines_axis(temp_marg.axes[0], ax_marg)
    _clone_stream_axis(temp_stream.axes[0], ax_stream)

    ax_marg.set_ylabel("Parameter")
    ax_stream.set_ylabel("Contribution")

    plt.close(temp_dag)
    plt.close(temp_marg)
    plt.close(temp_stream)

    arc_label = arc_edge.replace("->", "→")
    panel_label(ax_dag, "(a)  plot_dag")
    panel_label(ax_arc, f"(b)  plot_arcs  ({arc_label})")
    panel_label(ax_marg, f"(c)  plot_marginal  (node {target_name})")
    panel_label(ax_stream, f"(d)  plot_stream  (node {target_name})")

    return fig


gallery_fig = build_publication_gallery(model, target_node=TARGET_NODE, arc_edge=ARC_EDGE)

stem = "fig5_visualization-gallery"
pdf_path = FIGURES_DIR / f"{stem}.pdf"
png_path = FIGURES_DIR / f"{stem}.png"
gallery_fig.savefig(pdf_path)
gallery_fig.savefig(png_path, dpi=300)
print(f"Saved: {pdf_path}")
print(f"Saved: {png_path}")
gallery_fig